In [ ]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import os
import joblib

# --- GPU Überprüfung und Setup ---
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"NVIDIA GPU erkannt: {len(physical_devices)} GPU(s) verfügbar.")
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).")
    except RuntimeError as e:
        print(e)
else:
    print("Keine GPU erkannt. Training wird auf der CPU ausgeführt.")


df = pd.read_csv('training_data_ready.csv')
target_cols = ['x', 'y', 'z', 'Gewicht'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
X = df[x_cols]
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

scaler_dict = {
    #'standard': StandardScaler(),
    #'minmax': MinMaxScaler(),
    'robust': RobustScaler(),
    'maxabs': MaxAbsScaler(),
    #'quantile': QuantileTransformer(output_distribution='normal', random_state=42),
    #'power': PowerTransformer(method='yeo-johnson')
}

param_grid = {
    'scaler_name': ['robust', 'maxabs'], # ['standard', 'minmax', 'robust', 'maxabs', 'quantile', 'power']
    'hidden_layers': [
        #[128, 64, 32, 16],
        #[128, 64, 32],
        #[256, 128, 64],
        #[64, 32],
        [64, 32, 16],
        #[128, 64],
        [512, 256, 128, 64],
        #[32, 16]
    ],
    'l2_reg': [0.2, 0.1, 0.0],
    'dropout_rate': [0.1],#[0.0, 0.1, 0.2],
    'learning_rate': [0.001, 0.01, 0.005]
}

num_combinations = 100
random_search_configs = list(ParameterSampler(param_grid, n_iter=10000, random_state=42))

print(f"{len(random_search_configs)} Konfigurationen erfolgreich generiert.")

NVIDIA GPU erkannt: 1 GPU(s) verfügbar.
GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).
36 Konfigurationen erfolgreich generiert.


c:\Users\Adrian\anaconda3\envs\aki_tensor_2_10\lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 36 is smaller than n_iter=10000. Running 36 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [ ]:
os.makedirs('Modelle', exist_ok=True)

best_val_loss = float('inf')
best_model = None
best_scaler_name = None


all_results = []

configs_to_test = random_search_configs[148:]


for i, config in enumerate(random_search_configs):
    print(f"\n[{i}/{len(random_search_configs)-1}] Teste Konfiguration:")
    print(f" -> Scaler: {config['scaler_name']} | Netz: {config['hidden_layers']} | L2: {config['l2_reg']} | Dropout: {config['dropout_rate']} | LR: {config['learning_rate']}")

    current_scaler = scaler_dict[config['scaler_name']]
    X_train_scaled = current_scaler.fit_transform(X_train)

    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))

    for units in config['hidden_layers']:
        if config['l2_reg'] > 0:
            model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(config['l2_reg'])))
        else:
            model.add(layers.Dense(units, activation='relu'))

        if config['dropout_rate'] > 0:
            model.add(layers.Dropout(config['dropout_rate']))

    model.add(layers.Dense(4))

    optimizer = tf.keras.optimizers.Adam(learning_rate=config['learning_rate'])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=400,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_scaled, y_train,
        epochs=5000,
        batch_size=8,
        validation_data=(current_scaler.transform(X_val), y_val),
        callbacks=[early_stopping],
        verbose=0
    )

    min_val_loss = min(history.history['val_loss'])
    corresponding_mae = history.history['val_mae'][history.history['val_loss'].index(min_val_loss)]
    epochs_trained = len(history.history['val_loss'])

    print(f" -> Fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f}")

    layers_str = '-'.join(map(str, config['hidden_layers']))

    base_name = f"config_{i}_{config['scaler_name']}_layers_{layers_str}_l2_{config['l2_reg']}_drop_{config['dropout_rate']}_lr_{config['learning_rate']}_valloss_{min_val_loss:.4f}"
    model_path = f"Modelle/model_{base_name}.keras"
    model.save(model_path)

    scaler_path = f"Modelle/scaler_{base_name}.pkl"
    joblib.dump(current_scaler, scaler_path)

    all_results.append({
        'scaler': config['scaler_name'],
        'layers': str(config['hidden_layers']),
        'l2': config['l2_reg'],
        'dropout': config['dropout_rate'],
        'lr': config['learning_rate'],
        'val_loss': min_val_loss,
        'val_mae': corresponding_mae,
        'epochs': epochs_trained,
        'model_path': model_path,
        'scaler_path': scaler_path,
        'model_object': models
    })

    if min_val_loss < best_val_loss:
        best_val_loss = min_val_loss
        best_model = model
        best_scaler_name = config['scaler_name']

print("\n================ ARCHITEKTURSUCHE BEENDET ================")
all_results_sorted = sorted(all_results, key=lambda x: x['val_loss'])
print("\nDIE TOP 10 BESTEN ARCHITEKTUREN DIESES DURCHLAUFS:\n")
print(f"{'Platz':<6} | {'Scaler':<9} | {'Netz-Struktur':<18} | {'L2':<6} | {'Dropout':<7} | {'LR':<6} | {'Val-Loss (MSE)':<14} | {'Val-MAE':<8}")
print("-" * 95)
for rank, res in enumerate(all_results_sorted[:10], start=1):
    print(f"#{rank:<4} | {res['scaler']:<9} | {res['layers']:<18} | {res['l2']:<6} | {res['dropout']:<7} | {res['lr']:<6} | {res['val_loss']:<14.6f} | {res['val_mae']:<8.4f}")


[0/35] Teste Konfiguration:
 -> Scaler: robust | Netz: [64, 32, 16] | L2: 0.2 | Dropout: 0.1 | LR: 0.001
 -> Fertig nach 2174 Epochen. Bester Val-Loss: 128.8434

[1/35] Teste Konfiguration:
 -> Scaler: maxabs | Netz: [64, 32, 16] | L2: 0.2 | Dropout: 0.1 | LR: 0.001
 -> Fertig nach 3629 Epochen. Bester Val-Loss: 385.1770

[2/35] Teste Konfiguration:
 -> Scaler: robust | Netz: [64, 32, 16] | L2: 0.2 | Dropout: 0.1 | LR: 0.01
 -> Fertig nach 1001 Epochen. Bester Val-Loss: 143.4904

[3/35] Teste Konfiguration:
 -> Scaler: maxabs | Netz: [64, 32, 16] | L2: 0.2 | Dropout: 0.1 | LR: 0.01
 -> Fertig nach 1541 Epochen. Bester Val-Loss: 482.7406

[4/35] Teste Konfiguration:
 -> Scaler: robust | Netz: [64, 32, 16] | L2: 0.2 | Dropout: 0.1 | LR: 0.005
 -> Fertig nach 2324 Epochen. Bester Val-Loss: 95.7835

[5/35] Teste Konfiguration:
 -> Scaler: maxabs | Netz: [64, 32, 16] | L2: 0.2 | Dropout: 0.1 | LR: 0.005
 -> Fertig nach 2213 Epochen. Bester Val-Loss: 381.7123

[6/35] Teste Konfiguration:
 -

In [ ]:
final_scaler = scaler_dict[best_scaler_name]
X_test_scaled = final_scaler.transform(X_test)
test_eval = best_model.evaluate(X_test_scaled, y_test)
y_pred = best_model.predict(X_test_scaled)

y_pred_df = pd.DataFrame(
    y_pred,
    columns=['x_pred', 'y_pred', 'z_pred', 'Gewicht_pred'],
    index=y_test.index
)

results = y_test.copy()
results[['x_pred', 'y_pred', 'z_pred', 'Gewicht_pred']] = y_pred_df

results['source_file'] = df.loc[y_test.index, 'source_file'].values

results['abs_error_x'] = np.abs(results['x_pred'] - results['x'])
results['abs_error_y'] = np.abs(results['y_pred'] - results['y'])
results['abs_error_z'] = np.abs(results['z_pred'] - results['z'])
results['abs_error_Gewicht'] = np.abs(results['Gewicht_pred'] - results['Gewicht'])
results['mse_x'] = (results['x_pred'] - results['x']) ** 2
results['mse_y'] = (results['y_pred'] - results['y']) ** 2
results['mse_z'] = (results['z_pred'] - results['z']) ** 2
results['mse_Gewicht'] = (results['Gewicht_pred'] - results['Gewicht']) ** 2

print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

results_display = results[[
    'source_file',
    'x', 'x_pred', 'abs_error_x',
    'y', 'y_pred', 'abs_error_y',
    'z', 'z_pred', 'abs_error_z',
    'Gewicht', 'Gewicht_pred', 'abs_error_Gewicht'
]]
print("\nVorhersage vs. Realität für den Testdatensatz:")
print(results_display)

In [ ]:
y_pred = model.predict(X_train_scaled)

y_pred_df = pd.DataFrame(
    y_pred,
    columns=['x_pred', 'y_pred', 'z_pred'],
    index=y_train.index
)

results = y_train.copy()
results[['x_pred', 'y_pred', 'z_pred', 'Gewicht_pred']] = y_pred_df

results['source_file'] = df.loc[y_train.index, 'source_file'].values

results['abs_error_x'] = np.abs(results['x_pred'] - results['x'])
results['abs_error_y'] = np.abs(results['y_pred'] - results['y'])
results['abs_error_z'] = np.abs(results['z_pred'] - results['z'])
results['abs_error_Gewicht'] = np.abs(results['Gewicht_pred'] - results['Gewicht'])

results['mse_x'] = (results['x_pred'] - results['x']) ** 2
results['mse_y'] = (results['y_pred'] - results['y']) ** 2
results['mse_z'] = (results['z_pred'] - results['z']) ** 2
results['mse_Gewicht'] = (results['Gewicht_pred'] - results['Gewicht']) ** 2

print("Auswertung Trainingsdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

results_display = results[[
    'source_file',
    'x', 'x_pred', 'abs_error_x',
    'y', 'y_pred', 'abs_error_y',
    'z', 'z_pred', 'abs_error_z',
    'Gewicht', 'Gewicht_pred', 'abs_error_Gewicht'
]]
print("\nVorhersage vs. Realität für den Trainingsdatensatz:")
print(results_display)

In [ ]:
folder_path = "Modelle"

pattern = r"model_config_\d+_(?P<scaler>[^_]+)_layers_(?P<layers>[\d-]+)_l2_(?P<l2>[\d.]+)_drop_(?P<drop>[\d.]+)_lr_(?P<lr>[\d.]+)_valloss_(?P<valloss>[\d.]+).keras"

data = []

if os.path.exists(folder_path):
    for filename in os.listdir(folder_path):
        match = re.match(pattern, filename)
        if match:
            info = match.groupdict()
            
            data.append({
                "Modelgröße": info["layers"],
                "scaler": info["scaler"],
                "l2": float(info["l2"]),
                "dropout": float(info["drop"]),
                "lr": float(info["lr"]),
                "valloss": float(info["valloss"])
            })
else:
    print(f"Der Ordner '{folder_path}' wurde nicht gefunden.")

df = pd.DataFrame(data)

if not df.empty:
    print(f"{len(df)} passende Modell-Dateien gefunden.\n")
    pivot_table = df.pivot_table(
        index=["scaler", "l2", "dropout", "lr"],
        columns="Modelgröße",
        values="valloss",
        aggfunc=lambda x: ", ".join(map(str, sorted(x)))
    )

    print("--- Generierte Übersicht (MultiIndex) ---")
    print(pivot_table)
    
    excel_filename = "modell_uebersicht_hierarchisch.xlsx"
    pivot_table.to_excel(excel_filename, sheet_name="Modellübersicht", merge_cells=True, index=True, header=True, )
    print(f"\nDie Excel-Datei wurde erfolgreich als '{excel_filename}' gespeichert.")

else:
    print("Es wurden keine passenden Dateien im Ordner gefunden.")

307 passende Modell-Dateien gefunden.

--- Generierte Übersicht (MultiIndex) ---
Modelgröße                         128-64-32        128-64-32-16  \
scaler l2  dropout lr                                              
maxabs 0.0 0.0     0.001  369.5168, 378.5561  254.3418, 391.9625   
                   0.005  253.6405, 336.6367  291.1096, 310.5554   
                   0.010  330.1825, 341.8269  154.2244, 182.7614   
           0.1     0.001  317.1801, 372.0133  353.5575, 366.8224   
                   0.005            315.1795  339.7752, 347.8115   
                   0.010            257.3469  440.0678, 451.3312   
       0.1 0.0     0.001  384.5781, 396.8329  392.5718, 416.2447   
                   0.005  375.2041, 383.4933  268.2794, 362.6534   
                   0.010  199.1147, 368.7013  348.9325, 357.3714   
           0.1     0.001  371.5948, 400.4654  381.5393, 381.6712   
                   0.005  361.3845, 383.4676    406.075, 411.921   
                   0.010  314.5593,